# Generate the LR fixed-SIS backward matrix

Creates `sparse_grid_fracs_euclid_backward.pt`, mapping a 64×64 LR image-plane grid to a 64×64 source-plane grid. Coordinates are expressed in LR pixels, so the SIS deflection amplitude is `theta_E / lr_pixel_scale`.


In [ ]:
from pathlib import Path
import torch
import matplotlib.pyplot as plt
from differentiable_lensing import DifferentiableLensing

THETA_E_ARCSEC = 0.75
LR_PIXEL_SCALE_ARCSEC = 0.168
HR_PIXEL_SCALE_ARCSEC = 0.084
LR_GRID_SHAPE = 64
HR_GRID_SHAPE = 128
ALPHA_R_LR = THETA_E_ARCSEC / LR_PIXEL_SCALE_ARCSEC
ALPHA_R_HR = THETA_E_ARCSEC / HR_PIXEL_SCALE_ARCSEC
LOG_C = 4.5
DEVICE = "cpu"

print("alpha_r_lr [pixels]:", ALPHA_R_LR)
print("alpha_r_hr [pixels]:", ALPHA_R_HR)
assert abs(LR_PIXEL_SCALE_ARCSEC / 2 - HR_PIXEL_SCALE_ARCSEC) < 1e-12
assert HR_GRID_SHAPE == 2 * LR_GRID_SHAPE


In [ ]:
lens = DifferentiableLensing(DEVICE, target_resolution=1.0, target_shape=LR_GRID_SHAPE, alpha=None)
lower, upper = -LR_GRID_SHAPE / 2.0, LR_GRID_SHAPE / 2.0

source_x, source_y, theta_x_edges, theta_y_edges = lens.make_center_grid(
    lower, upper, LR_GRID_SHAPE
)
theta_x_edges = theta_x_edges.unsqueeze(0)
theta_y_edges = theta_y_edges.unsqueeze(0)

alpha = lens.construct_sis(theta_x_edges, theta_y_edges, ALPHA_R_LR)
beta_x_edges, beta_y_edges = lens.forward_lensing(theta_x_edges, theta_y_edges, alpha)

plt.figure(figsize=(5, 4))
plt.imshow(torch.sqrt(beta_x_edges[0]**2 + beta_y_edges[0]**2))
plt.title("LR source-plane radius of mapped image edges")
plt.colorbar()
plt.show()


In [ ]:
# Output cells are regular source cells; input cells are the lensed beta quadrilaterals.
grid_fracs = lens.square_grid_crop(
    source_x, source_y, beta_x_edges[0], beta_y_edges[0]
)
input_cell_areas = lens.nsq_As(beta_x_edges[0], beta_y_edges[0])
M, shape = lens.build_sparse_mapping(grid_fracs, input_cell_areas, DEVICE)
M = M.coalesce()

assert M.shape == (LR_GRID_SHAPE**2, LR_GRID_SHAPE**2)
assert torch.isfinite(M.values()).all()
torch.save(M, "sparse_grid_fracs_euclid_backward.pt")
print("saved", M.shape, "nnz", M._nnz(), "shape metadata", shape)
